# Tugas Praktikum 3 - Ekstraksi Fitur dan Splitting Data

**Nama:** Mohammad Al Hafis Hidayatulloh  
**NIM:** 254107023005  
**Mata Kuliah:** RTI235004 - Pembelajaran Mesin  
**Dosen Pengampu:** Usman Nurhasan, S.Kom., M.T.

---

## Deskripsi Tugas: Wisconsin Breast Cancer

Pada tugas praktikum ini saya menggunakan data `wbc.csv` (*Wisconsin Breast Cancer*) yang terdiri dari 569 data untuk mendiagnosis jenis kanker *Malignant* (`M` / ganas) dan *Benign* (`B` / jinak). Tahapan yang dikerjakan adalah:
1. Memisahkan antara variabel yang dapat digunakan dan variabel yang tidak dapat digunakan.
2. Melakukan proses encoding pada kolom `diagnosis`.
3. Melakukan proses standarisasi pada semua kolom yang memiliki nilai numerik.
4. Melakukan proses *stratified split* data untuk membuat data latih dan data uji dengan rasio 80:20.

### Langkah 0 - Load Data dan Inspeksi Awal

Memuat file `wbc.csv` dan mengecek informasi kolom serta tipe data untuk mengetahui kondisi seluruh variabel di dalam dataset.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df_raw = pd.read_csv('wbc.csv')
df_raw.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [2]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

Dari hasil `df_raw.info()` terlihat bahwa terdapat 569 baris dan 33 kolom. Dua kolom tidak dapat digunakan sebagai fitur maupun target pemodelan, yaitu:
- `id`: hanya nomor identitas unik rekam medis pasien (`int64`), tidak mencirikan karakteristik klinis sel kanker.
- `Unnamed: 32`: kolom kosong yang seluruh 569 barisnya bernilai `NaN` (`0 non-null`).

Sedangkan 31 kolom lainnya (`diagnosis` beserta 30 fitur numerik dari `radius_mean` hingga `fractal_dimension_worst`) terisi lengkap dan dapat digunakan.

### 1. Memisahkan Variabel yang Dapat Digunakan dan Tidak Dapat Digunakan

Memisahkan kolom `id` dan `Unnamed: 32` ke dalam `df_unused`, serta menyimpan 31 variabel yang digunakan ke dalam `df_wbc`.

In [3]:
# Daftar kolom yang tidak dapat digunakan
unused_cols = ['id', 'Unnamed: 32']

# Variabel yang tidak dapat digunakan
df_unused = df_raw[unused_cols].copy()

# Variabel yang dapat digunakan (diagnosis + 30 fitur numerik)
df_wbc = df_raw.drop(columns=unused_cols).copy()

print(f"Dimensi variabel yang digunakan (df_wbc)      : {df_wbc.shape}")
print(f"Dimensi variabel yang tidak digunakan (df_unused): {df_unused.shape}")
df_wbc.head()

Dimensi variabel yang digunakan (df_wbc)      : (569, 31)
Dimensi variabel yang tidak digunakan (df_unused): (569, 2)


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


### 2. Proses Encoding pada Kolom `diagnosis`

Menyimpan daftar nama 30 kolom numerik terlebih dahulu, lalu mengubah nilai kategorikal pada kolom `diagnosis` (`'B'` dan `'M'`) menjadi nilai numerik menggunakan `LabelEncoder`.

In [4]:
# Menyimpan nama ke-30 kolom numerik sebelum diagnosis di-encode
num_cols = df_wbc.select_dtypes(include=['float64', 'int64']).columns.tolist()
print(f"Jumlah kolom fitur numerik: {len(num_cols)} kolom")

# Encoding kolom diagnosis
le = LabelEncoder()
df_wbc['diagnosis'] = le.fit_transform(df_wbc['diagnosis'])

print("Mapping LabelEncoder diagnosis:", dict(zip(le.classes_, le.transform(le.classes_))))
print("\nDistribusi kelas diagnosis:")
print(df_wbc['diagnosis'].value_counts())
df_wbc.head()

Jumlah kolom fitur numerik: 30 kolom
Mapping LabelEncoder diagnosis: {'B': np.int64(0), 'M': np.int64(1)}

Distribusi kelas diagnosis:
diagnosis
0    357
1    212
Name: count, dtype: int64


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


Hasil encoding menunjukkan kategori `'B'` (*Benign* / jinak) diubah menjadi `0` sebanyak 357 data (`62.74%`), dan kategori `'M'` (*Malignant* / ganas) diubah menjadi `1` sebanyak 212 data (`37.26%`).

### 3. Proses Standarisasi pada Semua Kolom yang Memiliki Nilai Numerik

Menerapkan `StandardScaler` pada ke-30 kolom fitur numerik (`num_cols`) agar setiap fitur memiliki rata-rata ($\mu$) = 0 dan simpangan baku ($\sigma$) = 1.

In [5]:
scaler = StandardScaler()
df_wbc[num_cols] = scaler.fit_transform(df_wbc[num_cols])

df_wbc.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,1,1.097064,-2.073335,1.269934,0.984375,1.568466,3.283515,2.652874,2.532475,2.217515,2.255747,2.489734,-0.565265,2.833031,2.487578,-0.214002,1.316862,0.724026,0.660820,1.148757,0.907083,1.886690,-1.359293,2.303601,2.001237,1.307686,2.616665,2.109526,2.296076,2.750622,1.937015
1,1,1.829821,-0.353632,1.685955,1.908708,-0.826962,-0.487072,-0.023846,0.548144,0.001392,-0.868652,0.499255,-0.876244,0.263327,0.742402,-0.605351,-0.692926,-0.440780,0.260162,-0.805450,-0.099444,1.805927,-0.369203,1.535126,1.890489,-0.375612,-0.430444,-0.146749,1.087084,-0.243890,0.281190
2,1,1.579888,0.456187,1.566503,1.558884,0.942210,1.052926,1.363478,2.037231,0.939685,-0.398008,1.228676,-0.780083,0.850928,1.181336,-0.297005,0.814974,0.213076,1.424827,0.237036,0.293559,1.511870,-0.023974,1.347475,1.456285,0.527407,1.082932,0.854974,1.955000,1.152255,0.201391
3,1,-0.768909,0.253732,-0.592687,-0.764464,3.283553,3.402909,1.915897,1.451707,2.867383,4.910919,0.326373,-0.110409,0.286593,-0.288378,0.689702,2.744280,0.819518,1.115007,4.732680,2.047511,-0.281464,0.133984,-0.249939,-0.550021,3.394275,3.893397,1.989588,2.175786,6.046041,4.935010
4,1,1.750297,-1.151816,1.776573,1.826229,0.280372,0.539340,1.371011,1.428493,-0.009560,-0.562450,1.270543,-0.790244,1.273189,1.190357,1.483067,-0.048520,0.828471,1.144205,-0.361092,0.499328,1.298575,-1.466770,1.338539,1.220724,0.220556,-0.313395,0.613179,0.729259,-0.868353,-0.397100


In [6]:
# Cek ringkasan statistik hasil standarisasi
df_wbc[num_cols].describe().loc[['count', 'mean', 'std', 'min', 'max']]

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
count,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,569.000000,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,569.000000,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02,5.690000e+02
mean,-1.373633e-16,6.868164e-17,-1.248757e-16,-2.185325e-16,-8.366672e-16,1.873136e-16,4.995028e-17,-4.995028e-17,1.748260e-16,4.745277e-16,2.372638e-16,-1.123881e-16,-1.123881e-16,-1.311195e-16,-1.529727e-16,1.748260e-16,1.623384e-16,0.000000,8.741299e-17,-6.243785e-18,-8.241796e-16,1.248757e-17,-3.746271e-16,0.000000,-2.372638e-16,-3.371644e-16,7.492542e-17,2.247763e-16,2.622390e-16,-5.744282e-16
std,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00,1.000880e+00
min,-2.029648e+00,-2.229249e+00,-1.984504e+00,-1.454443e+00,-3.112085e+00,-1.610136e+00,-1.114873e+00,-1.261820e+00,-2.744117e+00,-1.819865e+00,-1.059924e+00,-1.554264e+00,-1.044049e+00,-7.378291e-01,-1.776065e+00,-1.298098e+00,-1.057501e+00,-1.913447,-1.532890e+00,-1.096968e+00,-1.726901e+00,-2.223994e+00,-1.693361e+00,-1.222423,-2.682695e+00,-1.443878e+00,-1.305831e+00,-1.745063e+00,-2.160960e+00,-1.601839e+00
max,3.971288e+00,4.651889e+00,3.976130e+00,5.250529e+00,4.770911e+00,4.568425e+00,4.243589e+00,3.927930e+00,4.484751e+00,4.910919e+00,8.906909e+00,6.655279e+00,9.461986e+00,1.104184e+01,8.029999e+00,6.143482e+00,1.207268e+01,6.649601,7.071917e+00,9.851593e+00,4.094189e+00,3.885905e+00,4.287337e+00,5.930172,3.955374e+00,5.112877e+00,4.700669e+00,2.685877e+00,6.046041e+00,6.846856e+00


### 4. Proses Stratified Split Data (Rasio 80:20)

Membagi data `df_wbc` menjadi data latih (`80%`) dan data uji (`20%`) menggunakan `train_test_split` dengan parameter `stratify=df_wbc['diagnosis']` supaya rasio kelas *Benign* (`0`) dan *Malignant* (`1`) pada data latih maupun data uji tetap terjaga sama seperti data aslinya.

In [7]:
# Stratified split 80:20 untuk membuat data latih dan data uji
df_train, df_test = train_test_split(
    df_wbc,
    test_size=0.2,
    random_state=0,
    stratify=df_wbc['diagnosis']
)

# Cek ukuran masing-masing bagian
print(f"Jumlah data asli : {df_wbc.shape[0]} baris")
print(f"Jumlah data latih: {df_train.shape[0]} baris ({df_train.shape[0]/df_wbc.shape[0]*100:.1f}%)")
print(f"Jumlah data uji  : {df_test.shape[0]} baris ({df_test.shape[0]/df_wbc.shape[0]*100:.1f}%)")
print("=" * 45)

# Cek jumlah dan persentase tiap label pada data asli, latih, dan uji
for nama, subset in [("Data Asli", df_wbc), ("Data Latih (80%)", df_train), ("Data Uji (20%)", df_test)]:
    counts = subset['diagnosis'].value_counts()
    props = subset['diagnosis'].value_counts(normalize=True) * 100
    print(f"\nDistribusi label {nama}:")
    for label in [0, 1]:
        ket = "Benign (0)" if label == 0 else "Malignant (1)"
        print(f"  - {ket:<14}: {counts[label]:>3} data ({props[label]:.2f}%)")

Jumlah data asli : 569 baris
Jumlah data latih: 455 baris (80.0%)
Jumlah data uji  : 114 baris (20.0%)

Distribusi label Data Asli:
  - Benign (0)    : 357 data (62.74%)
  - Malignant (1) : 212 data (37.26%)

Distribusi label Data Latih (80%):
  - Benign (0)    : 285 data (62.64%)
  - Malignant (1) : 170 data (37.36%)

Distribusi label Data Uji (20%):
  - Benign (0)    :  72 data (63.16%)
  - Malignant (1) :  42 data (36.84%)


In [8]:
# Memisahkan fitur (X) dan target (y) pada data latih dan data uji
X_train = df_train.drop(columns=['diagnosis'])
y_train = df_train['diagnosis']
X_test = df_test.drop(columns=['diagnosis'])
y_test = df_test['diagnosis']

print(f"Shape X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Shape X_test : {X_test.shape}, y_test : {y_test.shape}")

Shape X_train: (455, 30), y_train: (455,)
Shape X_test : (114, 30), y_test : (114,)


Dari hasil *stratified split* dengan rasio 80:20 di atas, dari total 569 baris data dihasilkan:
- **Data Latih (`df_train`):** 455 baris (`80.0%`), yang terdiri dari 285 data kelas `0` (*Benign*, `62.64%`) dan 170 data kelas `1` (*Malignant*, `37.36%`).
- **Data Uji (`df_test`):** 114 baris (`20.0%`), yang terdiri dari 72 data kelas `0` (*Benign*, `63.16%`) dan 42 data kelas `1` (*Malignant*, `36.84%`).

Proporsi kelas pada data latih (`62.64% : 37.36%`) dan data uji (`63.16% : 36.84%`) terbukti sangat konsisten mengikuti proporsi kelas pada data asli (`62.74% : 37.26%`). Hal ini penting karena dataset `wbc.csv` memiliki jumlah kelas *Benign* yang lebih banyak daripada *Malignant*, sehingga penggunaan `stratify` mencegah terjadinya ketimpangan distribusi kelas saat pelatihan maupun pengujian model.

---

## Kesimpulan

Pada Tugas Praktikum 3 ini, dataset *Wisconsin Breast Cancer* (`wbc.csv`) telah melalui seluruh tahapan persiapan fitur dan pembagian data. Pertama, variabel yang tidak relevan (`id` dan `Unnamed: 32`) dipisahkan sehingga tersisa 31 kolom yang valid (1 kolom target `diagnosis` dan 30 fitur numerik). Kedua, kolom `diagnosis` telah di-encode menggunakan `LabelEncoder` menjadi `0` (*Benign*) dan `1` (*Malignant*). Ketiga, seluruh 30 fitur numerik telah distandardisasi dengan `StandardScaler` sehingga memiliki skala yang seragam ($\mu = 0, \sigma = 1$). Terakhir, pembagian data menggunakan *stratified split* 80:20 berhasil membagi dataset menjadi 455 data latih dan 114 data uji dengan tetap mempertahankan rasio kelas sekitar 63% *Benign* berbanding 37% *Malignant* di kedua bagian.